# Extracción y Diagnóstico de Datos de Establecimientos del MINEDUC

Este notebook automatiza la descarga de registros de establecimientos educativos desde el portal del MINEDUC de Guatemala utilizando Selenium, limpia la información y ejecuta un diagnóstico paso a paso de calidad de datos.

## 1. Importación de Librerías y Configuración Global

In [1]:
import os
import time
import glob
import re
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.action_chains import ActionChains

# Parámetros globales de búsqueda y almacenamiento
BASE_URL = "https://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/"
CARPETA_DESCARGAS = os.path.join(os.getcwd(), "descargas")
CSV_SALIDA = "mineduc_establecimientos_unificado.csv"

DEPARTAMENTOS = [
    "ALTA VERAPAZ", "BAJA VERAPAZ", "CHIMALTENANGO", "CHIQUIMULA",
    "CIUDAD CAPITAL", "EL PROGRESO", "ESCUINTLA", "GUATEMALA",
    "HUEHUETENANGO", "IZABAL", "JALAPA", "JUTIAPA", "PETEN",
    "QUETZALTENANGO", "QUICHE", "RETALHULEU", "SACATEPEQUEZ",
    "SAN MARCOS", "SANTA ROSA", "SOLOLA", "SUCHITEPEQUEZ",
    "TOTONICAPAN", "ZACAPA"
]

NIVELES = ["BASICO", "DIVERSIFICADO"]

## 2. Funciones Auxiliares para Web Scraping

In [2]:
def encontrar_select_por_nombre(driver, nombre_parcial):
    selects = driver.find_elements(By.TAG_NAME, "select")
    for sel in selects:
        name = sel.get_attribute("name")
        if name and nombre_parcial in name:
            return sel
    return None

def encontrar_boton_por_src(driver, src_parcial):
    inputs = driver.find_elements(By.XPATH, "//input[@type='image']")
    for inp in inputs:
        src = inp.get_attribute("src")
        if src and src_parcial in src:
            return inp
    return None

def hacer_scroll_hasta_elemento(driver, elemento):
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", elemento)
    time.sleep(0.5)

def clic_seguro(driver, elemento):
    try:
        elemento.click()
    except Exception:
        ActionChains(driver).move_to_element(elemento).click().perform()

def esperar_descarga(carpeta, timeout=30):
    archivos_iniciales = set(glob.glob(os.path.join(carpeta, "*.xls")))
    start = time.time()
    while time.time() - start < timeout:
        archivos_actuales = set(glob.glob(os.path.join(carpeta, "*.xls")))
        nuevos = archivos_actuales - archivos_iniciales
        if nuevos:
            return max(nuevos, key=os.path.getctime)
        time.sleep(1)
    raise TimeoutError("No se detectó la descarga del archivo .xls")

def leer_tabla_desde_archivo_xls(ruta_archivo):
    with open(ruta_archivo, 'r', encoding='latin1') as f:
        contenido = f.read()
    tablas = pd.read_html(contenido)
    if not tablas:
        raise ValueError("No se encontraron tablas en el archivo")
    df = max(tablas, key=lambda t: t.shape[0])
    df.columns = df.iloc[0]
    df = df[1:].reset_index(drop=True)
    df = df.dropna(how='all')
    return df

## 3. Extracción y Carga Inicial de Datos

In [3]:
def descargar_departamento_nivel(driver, departamento, nivel):
    driver.get(BASE_URL)
    wait = WebDriverWait(driver, 15)

    select_depto = encontrar_select_por_nombre(driver, "cmbDepartamento")
    if not select_depto:
        raise Exception("No se encontró el select de Departamento")
    Select(select_depto).select_by_visible_text(departamento)
    time.sleep(1.5)

    select_nivel = encontrar_select_por_nombre(driver, "cmbNivel")
    if not select_nivel:
        raise Exception("No se encontró el select de Nivel")
    Select(select_nivel).select_by_visible_text(nivel)
    time.sleep(0.5)

    boton_buscar = driver.find_element(By.XPATH, "//input[@type='image' and contains(@src, 'seleccionar_estab.gif')]")
    hacer_scroll_hasta_elemento(driver, boton_buscar)
    clic_seguro(driver, boton_buscar)
    time.sleep(4)

    try:
        wait.until(EC.presence_of_element_located((By.XPATH, "//table")))
    except TimeoutException:
        print(f"  Sin resultados para {departamento} - {nivel}")
        return None

    boton_excel = encontrar_boton_por_src(driver, "export_excel.gif")
    if not boton_excel:
        try:
            boton_excel = driver.find_element(By.XPATH, "//*[contains(text(),'Generar Archivo de Excel')]")
        except Exception:
            pass
    if not boton_excel:
        raise Exception("No se encontró el botón de Excel")

    hacer_scroll_hasta_elemento(driver, boton_excel)
    wait.until(EC.element_to_be_clickable(boton_excel))
    boton_excel.click()

    ruta_archivo = esperar_descarga(CARPETA_DESCARGAS, timeout=30)
    nuevo_nombre = f"{departamento}_{nivel}.xls"
    nueva_ruta = os.path.join(CARPETA_DESCARGAS, nuevo_nombre)
    if os.path.exists(nueva_ruta):
        base, ext = os.path.splitext(nuevo_nombre)
        nueva_ruta = os.path.join(CARPETA_DESCARGAS, f"{base}_{int(time.time())}{ext}")
    os.rename(ruta_archivo, nueva_ruta)
    print(f"  Descargado: {os.path.basename(nueva_ruta)}")
    return nueva_ruta

def descargar_y_unificar():
    os.makedirs(CARPETA_DESCARGAS, exist_ok=True)

    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    prefs = {
        "download.default_directory": CARPETA_DESCARGAS,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True
    }
    options.add_experimental_option("prefs", prefs)

    driver = webdriver.Chrome(options=options)
    archivos_descargados = []

    try:
        print("Iniciando descarga de todos los departamentos (Básico y Diversificado)...")
        for depto in DEPARTAMENTOS:
            for nivel in NIVELES:
                try:
                    print(f"Procesando {depto} - {nivel}...")
                    ruta = descargar_departamento_nivel(driver, depto, nivel)
                    if ruta:
                        archivos_descargados.append(ruta)
                except Exception as e:
                    print(f"  ERROR: {e}")
                time.sleep(2)

        if not archivos_descargados:
            raise SystemExit("No se descargó ningún archivo.")

        dataframes = []
        for archivo in archivos_descargados:
            try:
                df = leer_tabla_desde_archivo_xls(archivo)
                nombre_base = os.path.basename(archivo).replace(".xls", "")
                partes = nombre_base.split("_")
                if len(partes) >= 2:
                    df["DEPARTAMENTO_CONSULTADO"] = partes[0]
                    df["NIVEL_CONSULTADO"] = partes[1]
                dataframes.append(df)
            except Exception as e:
                print(f"Error al leer {archivo}: {e}")

        df_unificado = pd.concat(dataframes, ignore_index=True)
        df_unificado.columns = [str(c).strip().upper() for c in df_unificado.columns]

        if "CODIGO" in df_unificado.columns:
            df_unificado = df_unificado[df_unificado["CODIGO"].notna()]
            df_unificado = df_unificado[df_unificado["CODIGO"].astype(str).str.strip() != ""]

        categoricas = ["DEPARTAMENTO", "MUNICIPIO", "NIVEL", "SECTOR", "AREA",
                       "STATUS", "MODALIDAD", "JORNADA", "PLAN", "DEPARTAMENTAL"]
        for col in categoricas:
            if col in df_unificado.columns:
                df_unificado[col] = df_unificado[col].astype(str).str.upper().str.strip()
                df_unificado[col] = df_unificado[col].replace({"NAN": pd.NA, "NONE": pd.NA, "---": pd.NA})

        if "TELEFONO" in df_unificado.columns:
            df_unificado["TELEFONO"] = df_unificado["TELEFONO"].astype(str).str.replace(r"\.0$", "", regex=True)
            df_unificado["TELEFONO"] = df_unificado["TELEFONO"].replace({"nan": pd.NA, "<NA>": pd.NA})

        if "CODIGO" in df_unificado.columns:
            df_unificado["_no_nulos"] = df_unificado.notna().sum(axis=1)
            df_unificado = df_unificado.sort_values("_no_nulos", ascending=False).drop_duplicates(subset=["CODIGO"]).drop(columns="_no_nulos")

        df_unificado.to_csv(CSV_SALIDA, index=False, encoding="utf-8-sig")
        return df_unificado
    finally:
        driver.quit()

# Obtención del Dataset
if os.path.exists(CSV_SALIDA):
    print(f"Cargando archivo existente '{CSV_SALIDA}'...")
    df = pd.read_csv(CSV_SALIDA, encoding='utf-8-sig')
else:
    print("Descargando datos...")
    df = descargar_y_unificar()

Cargando archivo existente 'mineduc_establecimientos_unificado.csv'...


## 4. Diagnóstico de Calidad de Datos

### a. Número de registros y variables
**Descripción:** Evalúa la dimensión total del conjunto de datos identificando la cantidad total de filas (registros) y columnas (variables).

In [4]:
print("a. Número de registros y variables:")
print(f"   - Registros (filas): {df.shape[0]}")
print(f"   - Variables (columnas): {df.shape[1]}")

a. Número de registros y variables:
   - Registros (filas): 27413
   - Variables (columnas): 19


### b. Tipo de dato de cada variable
**Descripción:** Detalla el tipo de dato asignado a cada columna (e.g., `object`, `int64`, `float64`) para asegurar su correcta interpretación analítica.

In [5]:
print("b. Tipo de dato de cada variable:")
for col in df.columns:
    print(f"   - {col}: {df[col].dtype}")

b. Tipo de dato de cada variable:
   - CODIGO: object
   - DISTRITO: object
   - DEPARTAMENTO: object
   - MUNICIPIO: object
   - ESTABLECIMIENTO: object
   - DIRECCION: object
   - TELEFONO: object
   - SUPERVISOR: object
   - DIRECTOR: object
   - NIVEL: object
   - SECTOR: object
   - AREA: object
   - STATUS: object
   - MODALIDAD: object
   - JORNADA: object
   - PLAN: object
   - DEPARTAMENTAL: object
   - DEPARTAMENTO_CONSULTADO: object
   - NIVEL_CONSULTADO: object


### c. Cantidad y porcentaje de valores faltantes por variable
**Descripción:** Cuantifica la presencia de valores nulos o ausentes en cada variable, calculando su proporción respecto al total de registros.

In [6]:
print("c. Cantidad y porcentaje de valores faltantes por variable:")
faltantes = df.isna().sum()
total = len(df)
for col in df.columns:
    nulos = faltantes[col]
    pct = (nulos / total) * 100
    print(f"   - {col}: {nulos} ({pct:.1f}%)")

c. Cantidad y porcentaje de valores faltantes por variable:
   - CODIGO: 0 (0.0%)
   - DISTRITO: 1616 (5.9%)
   - DEPARTAMENTO: 0 (0.0%)
   - MUNICIPIO: 0 (0.0%)
   - ESTABLECIMIENTO: 7 (0.0%)
   - DIRECCION: 279 (1.0%)
   - TELEFONO: 2800 (10.2%)
   - SUPERVISOR: 1620 (5.9%)
   - DIRECTOR: 4634 (16.9%)
   - NIVEL: 0 (0.0%)
   - SECTOR: 0 (0.0%)
   - AREA: 0 (0.0%)
   - STATUS: 0 (0.0%)
   - MODALIDAD: 0 (0.0%)
   - JORNADA: 0 (0.0%)
   - PLAN: 0 (0.0%)
   - DEPARTAMENTAL: 0 (0.0%)
   - DEPARTAMENTO_CONSULTADO: 0 (0.0%)
   - NIVEL_CONSULTADO: 0 (0.0%)


### d. Cantidad de valores únicos por variable
**Descripción:** Determina la cardinalidad de cada columna mostrando cuántos valores distintivos o categorías contiene.

In [7]:
print("d. Cantidad de valores únicos por variable:")
for col in df.columns:
    print(f"   - {col}: {df[col].nunique()}")

d. Cantidad de valores únicos por variable:
   - CODIGO: 27413
   - DISTRITO: 2017
   - DEPARTAMENTO: 23
   - MUNICIPIO: 357
   - ESTABLECIMIENTO: 10607
   - DIRECCION: 14724
   - TELEFONO: 13278
   - SUPERVISOR: 1501
   - DIRECTOR: 11459
   - NIVEL: 2
   - SECTOR: 4
   - AREA: 3
   - STATUS: 5
   - MODALIDAD: 2
   - JORNADA: 6
   - PLAN: 13
   - DEPARTAMENTAL: 26
   - DEPARTAMENTO_CONSULTADO: 23
   - NIVEL_CONSULTADO: 2


### e. Registros duplicados exactos
**Descripción:** Cuenta las filas que son idénticas en todas sus columnas para detectar redundancia total en la información.

In [8]:
duplicados_exactos = df.duplicated().sum()
print(f"e. Registros duplicados exactos: {duplicados_exactos}")

e. Registros duplicados exactos: 0


### f. Variables con valores fuera de dominio o inconsistentes
**Descripción:** Contrasta las opciones existentes en variables categóricas clave contra un conjunto predefinido de valores válidos esperados.

In [9]:
print("f. Variables con valores fuera de dominio o inconsistentes:")
dominios_esperados = {
    "DEPARTAMENTO": sorted(set(DEPARTAMENTOS) | {"ALTA VERAPAZ", "BAJA VERAPAZ", "CIUDAD CAPITAL"}),
    "NIVEL": ["BASICO", "DIVERSIFICADO", "PRIMARIA", "PARVULOS", "PREPRIMARIA BILINGUE", "PRIMARIA DE ADULTOS"],
    "SECTOR": ["OFICIAL", "PRIVADO", "MUNICIPAL", "COOPERATIVA"],
    "AREA": ["URBANA", "RURAL"],
    "STATUS": ["ABIERTA", "CERRADA DEFINITIVAMENTE", "CERRADA TEMPORALMENTE", "TEMPORAL NOMBRAMIENTO"],
    "MODALIDAD": ["MONOLINGUE", "BILINGUE"],
    "JORNADA": ["MATUTINA", "VESPERTINA", "NOCTURNA", "DOBLE", "SIN JORNADA"],
    "PLAN": ["DIARIO(REGULAR)", "SABATINO", "DOMINICAL", "FIN DE SEMANA", "IRREGULAR", "MIXTO", "A DISTANCIA", "VIRTUAL A DISTANCIA", "SEMIPRESENCIAL (FIN DE SEMANA)", "SEMIPRESENCIAL (UN DÍA A LA SEMANA)"]
}
for col, esperados in dominios_esperados.items():
    if col in df.columns:
        valores_reales = set(df[col].dropna().unique())
        fuera = valores_reales - set(esperados)
        if fuera:
            print(f"   - {col}: valores fuera del dominio esperado: {fuera}")
        else:
            print(f"   - {col}: todos los valores están dentro del dominio esperado.")

f. Variables con valores fuera de dominio o inconsistentes:
   - DEPARTAMENTO: todos los valores están dentro del dominio esperado.
   - NIVEL: todos los valores están dentro del dominio esperado.
   - SECTOR: todos los valores están dentro del dominio esperado.
   - AREA: valores fuera del dominio esperado: {'SIN ESPECIFICAR'}
   - STATUS: valores fuera del dominio esperado: {'TEMPORAL TITULOS'}
   - MODALIDAD: todos los valores están dentro del dominio esperado.
   - JORNADA: valores fuera del dominio esperado: {'INTERMEDIA'}
   - PLAN: valores fuera del dominio esperado: {'SEMIPRESENCIAL (DOS DÍAS A LA SEMANA)', 'INTERCALADO', 'SEMIPRESENCIAL'}


### g. Variables con formatos inconsistentes
**Descripción:** Inspecciona la sintaxis de campos específicos (e.g., código institucional y teléfonos) para validar que cumplan con la estructura estandarizada.

In [10]:
print("g. Variables con formatos inconsistentes:")

# Evaluación del patrón de CODIGO
inconsistentes = []
if "CODIGO" in df.columns:
    codigos = df["CODIGO"].dropna().astype(str)
    patron = re.compile(r'^\d{2}-\d{2}-\d{4}-\d{2}$')
    inconsistentes = codigos[~codigos.str.match(patron)]
    if len(inconsistentes) > 0:
        print(f"   - CODIGO: {len(inconsistentes)} registros no siguen el formato ##-##-####-##")
        print(f"     Ejemplos: {inconsistentes.head(3).tolist()}")
    else:
        print("   - CODIGO: todos los códigos siguen el formato esperado.")

# Evaluación de TELEFONO
no_numericos = []
if "TELEFONO" in df.columns:
    telefonos = df["TELEFONO"].dropna().astype(str)
    limpios = telefonos.str.replace(r'[\s\-\(\)]', '', regex=True)
    no_numericos = limpios[~limpios.str.isdigit()]
    if len(no_numericos) > 0:
        print(f"   - TELEFONO: {len(no_numericos)} registros contienen caracteres no numéricos.")
        print(f"     Ejemplos: {no_numericos.head(3).tolist()}")
    else:
        print("   - TELEFONO: todos los valores son numéricos (tras limpieza).")

g. Variables con formatos inconsistentes:
   - CODIGO: todos los códigos siguen el formato esperado.
   - TELEFONO: 99 registros contienen caracteres no numéricos.
     Ejemplos: ['55074625,58722678', '22206556Y22516346', '22534814y22381691']


### h. Identificación de problemas potenciales de calidad de datos
**Descripción:** Sintetiza los hallazgos principales de las pruebas previas y consolida los riesgos potenciales encontrados en la base de datos.

In [11]:
print("h. Identificación de problemas potenciales de calidad de datos:")
problemas = []

altos_faltantes = [col for col in df.columns if (df[col].isna().sum()/len(df))*100 > 50]
if altos_faltantes:
    problemas.append(f"   - Columnas con más del 50% de datos faltantes: {altos_faltantes}")
if duplicados_exactos > 0:
    problemas.append(f"   - Existen {duplicados_exactos} filas duplicadas exactas.")
if any(fuera for col, fuera in dominios_esperados.items() if col in df.columns):
    problemas.append("   - Algunas variables categóricas tienen valores fuera del dominio esperado (ver sección f).")
if "CODIGO" in df.columns and len(inconsistentes) > 0:
    problemas.append(f"   - {len(inconsistentes)} códigos tienen formato inesperado.")
if "TELEFONO" in df.columns and len(no_numericos) > 0:
    problemas.append(f"   - {len(no_numericos)} teléfonos contienen caracteres no numéricos.")

if problemas:
    for p in problemas:
        print(p)
else:
    print("   - No se detectaron problemas graves de calidad de datos.")

h. Identificación de problemas potenciales de calidad de datos:
   - Algunas variables categóricas tienen valores fuera del dominio esperado (ver sección f).
   - 99 teléfonos contienen caracteres no numéricos.


## 5. Plan de Limpieza

### 5.1 Descripción general del conjunto de datos

El conjunto de datos crudo (mineduc_establecimientos_unificado.csv) contiene el registro de establecimientos educativos de Guatemala en los niveles BÁSICO y DIVERSIFICADO, para los 23 departamentos consultados en el buscador del MINEDUC.

- Registros (filas): 27,413
- Variables (columnas): 19
- Tipo de dato de todas las variables: object (texto). Esto no significa que todas deban seguir siendo texto libre: entre ellas hay identificadores (CODIGO, DISTRITO), variables categóricas que deberían convertirse a tipo category una vez normalizadas (DEPARTAMENTO, MUNICIPIO, NIVEL, SECTOR, AREA, STATUS, MODALIDAD, JORNADA, PLAN, DEPARTAMENTAL), texto libre que debe permanecer como cadena normalizada (ESTABLECIMIENTO, DIRECCION, SUPERVISOR, DIRECTOR) y un campo de contacto (TELEFONO) que debe seguir siendo texto (no numérico), porque puede contener ceros a la izquierda o más de un número en la misma celda.
- Duplicados exactos: 0 (ya se eliminaron duplicados exactos por CODIGO durante la unificación de los archivos descargados).
- Valores faltantes nativos (NaN): concentrados en DIRECTOR (16.9%), TELEFONO (10.2%), DISTRITO y SUPERVISOR (5.9% cada una), DIRECCION (1.0%) y ESTABLECIMIENTO (0.03%).
- Valores faltantes "disfrazados" (no son NaN pero funcionan como tal): se detectaron marcadores de texto como ".", "-", "SIN DATO" y "S/D" en DIRECTOR (95 registros) y DIRECCION (10 registros), que deben tratarse como faltantes reales.
- Consistencia entre variables: se verificó que DEPARTAMENTO vs. DEPARTAMENTO_CONSULTADO, NIVEL vs. NIVEL_CONSULTADO, y el prefijo departamental de CODIGO vs. DEPARTAMENTO, coinciden al 100% (0 discrepancias). Esto confirma que la unión de los archivos por departamento/nivel fue correcta, pero también indica que DEPARTAMENTO_CONSULTADO y NIVEL_CONSULTADO son columnas redundantes (candidatas a eliminarse o documentarse como metadatos de trazabilidad de la extracción).

### 5.2 Variables que más operaciones de limpieza necesitarán

| Prioridad | Variable | Motivo principal |
|---|---|---|
| 1 | TELEFONO | 10.2% de nulos, longitudes muy dispares (6 a 26 caracteres), 84 celdas con más de un número separado por coma o "Y", 99 registros con caracteres no numéricos. |
| 2 | DIRECTOR | Mayor tasa de faltantes reales (16.9%) más 95 valores placeholder ("." "-" "SIN DATO" "S/D") que elevan el faltante real a ~17.3%. |
| 3 | ESTABLECIMIENTO | Variable central para reportes y para la detección de duplicados parciales; tiene inconsistencias de puntuación (comillas simples de nombres de origen maya escritas con comilla invertida en vez de apóstrofe, uso de guion bajo en vez de paréntesis) y abreviaturas institucionales inconsistentes (p. ej. "INEB" vs. "INSTITUTO NACIONAL DE EDUCACIÓN BÁSICA"). |
| 4 | DIRECCION | Texto libre sin estructura fija, 26 registros no están en mayúsculas (inconsistentes con el resto), 10 placeholders de faltante, abreviaturas dispares (KM., AVE., ZONA). |
| 5 | DISTRITO | Formato de código inconsistente: coexisten longitudes de 3, 6 y 10 caracteres para el mismo departamento, sin un patrón claro. |
| 6 | MUNICIPIO | Requiere validación contra el catálogo oficial de municipios/zonas; en CIUDAD CAPITAL el campo contiene "ZONA 1"…"ZONA 25" en vez de un municipio (convención válida del MINEDUC, pero debe verificarse contra dominio permitido). |
| 7 | AREA, STATUS, JORNADA, PLAN | Categorías fuera del dominio inicialmente esperado (SIN ESPECIFICAR, TEMPORAL TITULOS, INTERMEDIA, SEMIPRESENCIAL (DOS DÍAS A LA SEMANA), INTERCALADO, SEMIPRESENCIAL); en realidad son categorías válidas del catálogo del MINEDUC que faltaban en el dominio de referencia usado en el diagnóstico. |
| 8 | SUPERVISOR | 5.9% de faltantes reales; requiere la misma normalización ortográfica de texto libre que DIRECTOR. |
| 9 | DEPARTAMENTO_CONSULTADO, NIVEL_CONSULTADO | Redundantes (100% iguales a DEPARTAMENTO y NIVEL); decidir si se eliminan o se conservan como metadato de extracción. |

Las variables categóricas ya limpias en cuanto a dominio (NIVEL, SECTOR, MODALIDAD, CODIGO, DEPARTAMENTO) requieren pasos de limpieza mínimos (verificación de formato/tipo), pero se incluyen igual en la sección 5.3 porque la guía del proyecto pide describir la estrategia para todas las variables.

Principio general (aplica a todas las variables de texto libre): no se eliminarán tildes, ñ, ni se aplicarán transformaciones agresivas de mayúsculas/minúsculas que alteren la ortografía correcta de nombres de personas, establecimientos o direcciones. La limpieza de texto se limita a: espacios (inicio/fin/dobles), caracteres invisibles, signos de puntuación mal usados como sustitutos de otros signos (por ejemplo la comilla invertida o el guion bajo), y estandarización de mayúsculas solo cuando el dato ya viene naturalmente en mayúsculas en el resto del dataset (para no introducir inconsistencia). El objetivo es que el dato quede bien escrito para que analistas y redactores de informes puedan citarlo directamente.

### 5.3 Estrategia de limpieza por variable

#### CODIGO (identificador único del establecimiento)

1. Problemas encontrados: ninguno grave. El 100% de los códigos siguen el patrón ##-##-####-## y no hay duplicados de CODIGO.
2. Regla y por qué funcionará: se mantiene como texto (no numérico) porque tiene ceros a la izquierda y guiones con significado estructural (departamento-municipio-secuencia-nivel); solo se aplicará strip() por higiene y se validará el patrón con una expresión regular como prueba automática de calidad. Funciona porque ya se confirmó que el 100% de los valores cumplen el formato esperado.
3. Riesgos: ninguno relevante; el único riesgo sería convertirlo a número y perder los ceros a la izquierda o el significado posicional de cada segmento.

#### DISTRITO

1. Problemas encontrados: formato inconsistente. Coexisten longitudes de 3, 6 y 10 caracteres (p. ej. "003", "01-501", "01-01-0007") incluso dentro del mismo departamento, y 5.9% de valores faltantes.
2. Regla y por qué funcionará: normalizar a texto con strip(), documentar los tres patrones encontrados en el Libro de Códigos como variantes válidas (no se puede inferir un único formato canónico sin el catálogo oficial de distritos del MINEDUC), y dejar los faltantes como NA explícito. No se intentará "adivinar" o completar distritos faltantes porque no hay una regla determinística confiable a partir de las demás columnas.
3. Riesgos: si se fuerza un único formato (por ejemplo, rellenando con ceros a 10 caracteres) sin conocer la codificación oficial, se podría introducir un código inválido que no corresponde a ningún distrito real.

#### DEPARTAMENTO

1. Problemas encontrados: ninguno; los 23 valores están dentro del dominio esperado y coinciden 100% con DEPARTAMENTO_CONSULTADO y con el prefijo de CODIGO.
2. Regla y por qué funcionará: ya viene en mayúsculas y sin espacios extra (verificado); se aplicará de todas formas strip() + colapso de espacios múltiples como medida preventiva, y se convertirá a tipo category con dominio fijo de 22 departamentos de Guatemala (CIUDAD CAPITAL se documentará como la subdivisión que usa el MINEDUC para el departamento de Guatemala). Funciona porque la validación de dominio ya mostró 0 valores fuera de catálogo.
3. Riesgos: bajo. El único riesgo es tratar CIUDAD CAPITAL como un departamento aparte de GUATEMALA en análisis geográficos que usen el catálogo oficial de 22 departamentos; debe documentarse la equivalencia.

#### MUNICIPIO

1. Problemas encontrados: para DEPARTAMENTO = CIUDAD CAPITAL, el campo contiene "ZONA 1"…"ZONA 25" en vez de un nombre de municipio real (4,155 registros). El resto de departamentos tiene entre 5 y 33 municipios distintos, un rango creíble frente al catálogo oficial del INE, pero no se ha contrastado registro a registro contra dicho catálogo.
2. Regla y por qué funcionará: normalizar texto (mayúsculas ya consistentes, strip(), espacios múltiples) y contrastar cada valor contra el catálogo oficial de municipios de Guatemala (por departamento); las filas con MUNICIPIO tipo "ZONA N" se dejarán tal cual pero marcadas con una bandera/derivada ES_ZONA_CIUDAD_CAPITAL en vez de intentar mapearlas a un municipio que no existe, ya que la Ciudad Capital no se subdivide en municipios sino en zonas dentro del municipio de Guatemala. Funciona porque preserva la granularidad real de la fuente sin inventar un municipio falso.
3. Riesgos: si se fuerza MUNICIPIO = "GUATEMALA" para las zonas de Ciudad Capital se pierde la granularidad geográfica (zona) sin necesidad; si se deja sin marcar, un análisis automatizado contra el catálogo de municipios reportará falsos "municipios inválidos".

#### DEPARTAMENTAL (oficina regional/departamental del MINEDUC que supervisa)

1. Problemas encontrados: ninguno estructural. Se observó que un mismo DEPARTAMENTO puede tener varias DEPARTAMENTAL (p. ej. Guatemala tiene 4: Norte, Sur, Central, Oriente), lo cual es correcto porque las direcciones departamentales de educación no siempre coinciden 1 a 1 con el departamento administrativo.
2. Regla y por qué funcionará: normalizar texto (strip(), mayúsculas, espacios múltiples) y convertir a category; documentar explícitamente en el Libro de Códigos que esta variable representa una oficina administrativa del MINEDUC y no el departamento geográfico, para evitar que se use por error como sinónimo de DEPARTAMENTO.
3. Riesgos: riesgo de confusión analítica (no de limpieza) si alguien asume que DEPARTAMENTAL y DEPARTAMENTO deben ser siempre iguales; se mitiga documentando la relación real.

#### DEPARTAMENTO_CONSULTADO y NIVEL_CONSULTADO

1. Problemas encontrados: son 100% redundantes respecto a DEPARTAMENTO y NIVEL (0 discrepancias en 27,413 filas). Se originaron como metadato interno del proceso de scraping (para saber con qué filtro de búsqueda se descargó cada archivo).
2. Regla y por qué funcionará: se eliminarán del conjunto de datos limpio final, documentando en el registro de transformaciones que su función (trazabilidad de la extracción) ya está garantizada por la coincidencia perfecta con DEPARTAMENTO/NIVEL, y su presencia solo añade redundancia sin aportar información nueva.
3. Riesgos: bajo; el único riesgo es que, si en el futuro se vuelve a extraer el dataset y aparece una fila donde el filtro de búsqueda no coincide con el valor real (por ejemplo, un error de clasificación del MINEDUC), se perdería esa señal de auditoría. Se documentará la decisión para poder revertirla si llegara a presentarse ese caso.

#### ESTABLECIMIENTO

1. Problemas encontrados: 7 valores faltantes (0.03%); 7 registros con puntuación mal usada como sustituto de otros signos (comillas de apóstrofe de origen maya escritas con comilla invertida en vez de apóstrofe recto, y guion bajo usado en vez de paréntesis, p. ej. "...DE EDUCACIÓN MEDIA_CEPMEM"); nombres institucionales muy repetidos de forma genérica ("INEB DE TELESECUNDARIA" aparece 1,776 veces, "INEB" 681 veces) que son nombres legítimos de muchas escuelas distintas, no duplicados; posibles duplicados parciales cuando el mismo nombre se repite junto con la misma MUNICIPIO y DIRECCION (11,118 filas), la mayoría explicadas porque un mismo plantel ofrece BÁSICO y DIVERSIFICADO bajo dos CODIGO distintos.
2. Regla y por qué funcionará:
   - Se aplicará únicamente strip() y colapso de espacios múltiples (no se toca mayúsculas/minúsculas: la variable ya es 100% mayúsculas y así se usa en los reportes oficiales del MINEDUC).
   - Se normalizará la puntuación anómala reemplazando la comilla invertida por el apóstrofe recto y el guion bajo usado como paréntesis por "(" ")" solo en los 7 casos puntuales identificados manualmente (no con un reemplazo masivo, porque la comilla invertida también aparece dentro de nombres de origen maya como parte legítima de la ortografía, p. ej. "K'ULB' IL", y no debe eliminarse ni "corregirse" a una forma sin ese signo).
   - Para duplicados parciales se usará similitud de cadenas (Levenshtein/RapidFuzz) sobre ESTABLECIMIENTO + MUNICIPIO + DIRECCION, pero no se eliminará ningún registro automáticamente: cada grupo de alta similitud se revisará contra CODIGO y NIVEL para distinguir entre (a) el mismo plantel con oferta BÁSICO/DIVERSIFICADO (se conservan ambas filas) y (b) una posible reinscripción o error de captura (se documenta caso por caso).
3. Riesgos: el mayor riesgo es sobre-corregir la ortografía (quitar tildes, "normalizar" nombres de origen maya, o fusionar nombres solo por similitud textual) y terminar fusionando escuelas realmente distintas o dañando la ortografía que necesitan los redactores de informes. Por eso la limpieza de puntuación se hace dirigida (no con reemplazo global) y la deduplicación parcial es solo de detección, con decisión manual documentada.

#### DIRECCION

1. Problemas encontrados: 1.0% de faltantes nativos + 10 registros con placeholders (".", "-") que en realidad son faltantes; 26 registros no están en mayúsculas (inconsistentes con el resto del campo); texto libre con abreviaturas dispares ("AVE.", "AV.", "KM.", "ZONA") y sin estructura fija.
2. Regla y por qué funcionará: convertir los placeholders (".", "-", y variantes como "N/A", "SIN DATO") a NA explícito; homologar a mayúsculas los 26 registros discordantes para que la variable sea consistente con el 99.9% restante (no se trata de imponer mayúsculas arbitrariamente, sino de igualar el formato dominante ya presente); aplicar strip() y colapso de espacios múltiples. No se estandarizarán las abreviaturas de tipo de vía (AVE/AV/KM) a una única forma, porque no hay un catálogo de direcciones postales fiable para Guatemala rural y forzarlo generaría más ambigüedad que la que resuelve.
3. Riesgos: al convertir a mayúsculas los 26 registros restantes se pierde la distinción entre mayúsculas/minúsculas si esa distinción fuera intencional (por ejemplo, un nombre propio); se mitiga revisando manualmente esos 26 casos antes de aplicar el cambio masivo. Convertir placeholders a NA sin revisión podría ocultar un caso donde "-" es parte real de una dirección (p. ej. un lote con guion); se mitiga limitando la regla a valores que son exactamente el placeholder tras strip(), no una subcadena.

#### SUPERVISOR

1. Problemas encontrados: 5.9% de valores faltantes; no se detectaron placeholders textuales disfrazados de dato (a diferencia de DIRECTOR); no se detectaron problemas de mayúsculas.
2. Regla y por qué funcionará: strip() + colapso de espacios múltiples como higiene preventiva; se conserva la ortografía completa (tildes, tratamientos "DE", tildes en apellidos compuestos) sin alterarla. Se aplicará similitud de cadenas contra DIRECTOR y contra sí misma solo para detectar posibles errores de tecleo en un mismo nombre repetido con leves variaciones (no para fusionar automáticamente).
3. Riesgos: riesgo bajo; el principal riesgo sería "corregir" una variante ortográfica real de un nombre poco común pensando que es un error de tecleo (dos supervisores distintos con apellidos parecidos). Se documentará cada caso detectado por similitud antes de decidir si se unifica.

#### DIRECTOR

1. Problemas encontrados: mayor tasa de faltantes reales (16.9%) más 95 valores placeholder ("." 37, "-" 29, "SIN DATO" 28, "S/D" 1) que deben tratarse como faltantes adicionales (faltante real total ≈ 17.3%).
2. Regla y por qué funcionará: convertir los placeholders detectados a NA explícito (regla exacta tras strip() + mayúsculas, igual que en DIRECCION); strip() y colapso de espacios múltiples; conservar tildes y ortografía completa de los nombres. No se imputará el nombre del director faltante bajo ninguna circunstancia (no existe una fuente confiable para inferirlo).
3. Riesgos: con casi 1 de cada 5 registros sin director real, cualquier análisis que dependa de esta variable debe reportar explícitamente esa tasa de faltantes para no subestimar la incertidumbre; convertir placeholders a NA de forma exacta (no por subcadena) evita el riesgo de borrar accidentalmente un nombre real que por coincidencia contenga un guion.

#### TELEFONO

1. Problemas encontrados: 10.2% de faltantes; longitudes muy dispares (8 dígitos en el 88% de los casos, pero también 6, 7, 11, 14, 15, 16, 17, 18 y 26 caracteres); 84 celdas contienen más de un número de teléfono separados por coma o por la letra "Y"/"y" (p. ej. "55074625, 58722678", "22206556 Y 2251-6346"); 99 registros con caracteres no numéricos (letras, guiones internos).
2. Regla y por qué funcionará:
   - Se creará una variable derivada TELEFONO_LISTA (lista de números separados) a partir de dividir por los separadores detectados (coma, "y", "Y"), y una variable TELEFONO_PRINCIPAL con el primer número de esa lista, ambas documentadas en el Libro de Códigos con su justificación (permitir análisis que requieran un único teléfono por registro sin perder los números adicionales).
   - Cada número individual se limpiará quitando espacios y guiones internos y se validará contra el patrón de 8 dígitos usado en Guatemala; los que no cumplan ese patrón (muy cortos, muy largos, con letras) se dejarán como NA en TELEFONO_PRINCIPAL pero se conservará el valor original en TELEFONO sin modificar, para no perder el dato crudo.
   - La variable se mantiene como texto (no se convierte a numérico) para preservar ceros a la izquierda y evitar que pandas lo interprete como float.
3. Riesgos: dividir por comas/"Y" podría fallar si el separador no fue uno de los detectados (por ejemplo, un espacio simple entre dos números de 8 dígitos sin ninguna letra o coma), dejando ese caso sin dividir; se mitiga revisando manualmente los registros de longitud atípica (11, 14+ caracteres) que no coincidan con los patrones de separador conocidos. Validar contra el patrón de 8 dígitos podría descartar como inválidos números con formato antiguo o extensiones internas legítimas; por eso el dato original se conserva siempre y solo se marca como no-válida la versión derivada.

#### NIVEL y NIVEL_CONSULTADO

1. Problemas encontrados: ninguno; solo 2 valores (BASICO, DIVERSIFICADO), sin faltantes, 100% consistentes entre sí.
2. Regla y por qué funcionará: strip() preventivo y conversión a category. Funciona porque el dominio ya está validado al 100%.
3. Riesgos: ninguno.

#### SECTOR

1. Problemas encontrados: ninguno; 4 valores (OFICIAL, PRIVADO, MUNICIPAL, COOPERATIVA), todos dentro de dominio, sin faltantes.
2. Regla y por qué funcionará: strip() preventivo y conversión a category.
3. Riesgos: ninguno.

#### AREA

1. Problemas encontrados: aparece la categoría "SIN ESPECIFICAR" que no estaba en el dominio inicial de referencia (URBANA, RURAL) usado en el diagnóstico.
2. Regla y por qué funcionará: se amplía el dominio válido a {URBANA, RURAL, SIN ESPECIFICAR}, tratando "SIN ESPECIFICAR" como una categoría legítima de "dato no reportado por el establecimiento" (no se convierte a NA porque el propio MINEDUC la distingue de un campo vacío, y el diagnóstico mostró 0% de faltantes nativos en esta columna). Se documenta la razón en el Libro de Códigos.
3. Riesgos: si un analista futuro asume que el dominio es binario (URBANA/RURAL) sin leer el Libro de Códigos, podría descartar por error los registros "SIN ESPECIFICAR" en vez de tratarlos como una tercera categoría válida.

#### STATUS

1. Problemas encontrados: aparece "TEMPORAL TITULOS" fuera del dominio inicial de referencia.
2. Regla y por qué funcionará: se agrega "TEMPORAL TITULOS" como categoría válida adicional del catálogo real del MINEDUC (junto a ABIERTA, CERRADA DEFINITIVAMENTE, CERRADA TEMPORALMENTE, TEMPORAL NOMBRAMIENTO), documentando su significado (autorización temporal para emitir títulos) en el Libro de Códigos.
3. Riesgos: confundir "TEMPORAL TITULOS" con "TEMPORAL NOMBRAMIENTO" si no se documenta bien la diferencia; se mitiga con una definición explícita de cada categoría en el Libro de Códigos.

#### MODALIDAD

1. Problemas encontrados: ninguno; 2 valores (MONOLINGUE, BILINGUE), sin faltantes, dentro de dominio.
2. Regla y por qué funcionará: strip() preventivo y conversión a category.
3. Riesgos: ninguno.

#### JORNADA

1. Problemas encontrados: aparece "INTERMEDIA" fuera del dominio inicial de referencia.
2. Regla y por qué funcionará: se agrega "INTERMEDIA" como categoría válida adicional (jornada entre matutina y vespertina, usada por el MINEDUC), documentada en el Libro de Códigos junto con las demás 6 categorías observadas.
3. Riesgos: bajo; riesgo de que se confunda con "DOBLE" si no se documenta la diferencia.

#### PLAN

1. Problemas encontrados: aparecen 3 categorías fuera del dominio inicial de referencia: "SEMIPRESENCIAL", "SEMIPRESENCIAL (DOS DÍAS A LA SEMANA)" e "INTERCALADO".
2. Regla y por qué funcionará: se amplía el dominio de referencia con las 13 categorías reales observadas en los datos (en vez de la lista incompleta usada en el diagnóstico inicial), ya que todas corresponden a modalidades administrativas reales del MINEDUC y no a errores de captura. Se documentará cada una en el Libro de Códigos.
3. Riesgos: bajo; el riesgo sería fusionar por error "SEMIPRESENCIAL" con "SEMIPRESENCIAL (DOS DÍAS A LA SEMANA)" asumiendo que son la misma categoría cuando en realidad distinguen frecuencia; se mantienen separadas hasta confirmar con el catálogo oficial del MINEDUC.